# 01 — Basic Backtest

End-to-end sanity check: one ticker, one parameter pair, costs on.

The point of this notebook is to verify the engine wired up correctly before we do any analysis. If buy-and-hold doesn't match the price ratio and if the equity curve doesn't compose, nothing downstream matters.

In [ ]:
from dataclasses import asdict

import pandas as pd

from ma_backtester.backtester import run_backtest, run_buy_and_hold
from ma_backtester.benchmark import compare_strategies
from ma_backtester.config import CostConfig, StrategyConfig
from ma_backtester.costs import FixedBpsCost
from ma_backtester.data import load_close
from ma_backtester.metrics import compute_metrics_table
from ma_backtester.plotting import (
    equity_curve,
    install_theme,
    report_dashboard,
    signal_overlay,
)

install_theme()

## Load data

SPY adjusted close from yfinance (cached locally as parquet on first run).

In [ ]:
close = load_close("SPY", start="2010-01-01", end="2024-12-31")
print(f"{len(close)} bars from {close.index[0].date()} to {close.index[-1].date()}")
close.tail(3)

## Run the strategy and the benchmark

Strategy: SMA(20, 50) crossover. Benchmark: buy-and-hold. Both pay 5 bps per side so the comparison is apples-to-apples on cost.

In [ ]:
strategy = StrategyConfig(fast_window=20, slow_window=50)
cost = FixedBpsCost(CostConfig(per_side_bps=5.0))

strat = run_backtest(close=close, strategy_config=strategy, cost_model=cost)
bench = run_buy_and_hold(close=close, cost_model=cost)

strat_m = compute_metrics_table(
    equity=strat.equity,
    daily_returns=strat.daily_returns,
    positions=strat.positions,
    trades=strat.trades,
)
bench_m = compute_metrics_table(
    equity=bench.equity,
    daily_returns=bench.daily_returns,
    positions=bench.positions,
    trades=bench.trades,
)
pd.DataFrame({"strategy": asdict(strat_m), "buy_and_hold": asdict(bench_m)})

## Equity curve and dashboard

Log scale makes a constant CAGR a straight line, equalising visual weight across the period.

In [ ]:
equity_curve(strategy_equity=strat.equity, benchmark_equity=bench.equity)

In [ ]:
report_dashboard(
    strategy_equity=strat.equity,
    benchmark_equity=bench.equity,
    strategy_returns=strat.daily_returns,
)

## Trade ledger sanity

In [ ]:
strat.trades.head(8)

In [ ]:
signal_overlay(close=close, positions=strat.positions)

## Statistical comparison

CAPM regression with Newey-West HAC standard errors, information ratio, and the Memmel-corrected Jobson-Korkie Sharpe difference test.

If the strategy is just a beta-1 wrapper of the benchmark, alpha should be near zero and not statistically distinguishable from zero. Costs are a drag, so if anything we'd expect slightly negative alpha.

In [ ]:
cmp = compare_strategies(
    strategy_returns=strat.daily_returns,
    benchmark_returns=bench.daily_returns,
)
pd.Series(asdict(cmp))